### Imports

In [1]:
import json
import numpy as np
import pandas as pd
import pingouin as pg
import seaborn as sn

print(pg.__version__) # 0.5.3
print(pd.__version__) # 2.0.3
print(np.__version__) # 1.24.3
print(sn.__version__) # 0.13.0

from utils_MS import *

# %load_ext autotime

0.5.3
2.2.2
1.26.4
0.13.2


/home/ealvarez/miniconda3/envs/graph_matching/lib/python3.10/site-packages/outdated/utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.3, the latest is 0.6.1.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


In [2]:
# def run(exp):

### Parameters

In [3]:
file = open("exp.json")
experiment = json.load(file)
exp = experiment["exp"]

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

apply_transformation = params["apply_transformation"]
print("Apply transformation:", apply_transformation)

threshold_corr = params["threshold_corr"]
print("Threshold corr:\t", threshold_corr)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups_id_no = params["groups_id_no"]
print("Groups id (no):\t", groups_id_no)

Exp:		 exp30
Data variations: ['none']
Apply transformation: False
Threshold corr:	 0.5
Groups id:	 ['AR', 'BC', 'BPH', 'CKD-Mild', 'CKD-Moderate', 'CKD-Severe', 'CKD5-HD', 'CKD5-PD', 'CRS', 'Health', 'LPRD', 'LSNB', 'OSA', 'PCa', 'PD', 'RCC', 'SGB', 'SKD']
Subgroups id:	 {'AR': ['1', '2'], 'BC': ['1', '2'], 'BPH': ['1', '2'], 'CKD-Mild': ['1', '2'], 'CKD-Moderate': ['1', '2'], 'CKD-Severe': ['1', '2'], 'CKD5-HD': ['1', '2'], 'CKD5-PD': ['1', '2'], 'CRS': ['1', '2'], 'Health': ['1', '2'], 'LPRD': ['1', '2'], 'LSNB': ['1', '2'], 'OSA': ['1', '2'], 'PCa': ['1', '2'], 'PD': ['1', '2'], 'RCC': ['1', '2'], 'SGB': ['1', '2'], 'SKD': ['1', '2']}
Groups id (no):	 ['Blank', 'QC', 'Std']


In [4]:
# Remove
# groups_id = ["LSNB", "BC", "RCC", "BPH", "PD"] # "OSA" Memory issue
# groups_id = ["AR", "BC"]

### Load dataset

In [5]:
# read raw data
df_join_raw = pd.read_csv("experiments/input/{}_raw.csv".format(exp), index_col=0)
df_join_raw

,Average Rt,Average Mz,Metabolite name,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,...,SKD_2.43,SKD_2.44,SKD_2.45,SKD_2.46,SKD_2.47,SKD_2.48,SKD_2.49,SKD_2.50,SKD_2.51,SKD_2.52
0,1.0,69.99951,Unknown,0.786205,0.836262,-0.975441,0.067434,-0.905178,0.458622,-0.756562,...,-0.906900,0.153942,0.230629,-0.554414,-0.477171,0.569892,-0.270328,-0.399430,-0.774251,0.137557
1,1.0,70.04025,Unknown,2.395434,2.572471,2.052794,2.617854,2.349623,2.501793,2.941043,...,4.318123,1.830162,3.076034,2.600924,2.340488,2.455378,2.681824,3.990780,3.177152,2.460345
2,1.0,70.04151,Unknown,0.564810,1.888370,0.508263,1.088222,0.559000,1.711682,0.639700,...,0.433149,1.329261,1.390541,0.725222,0.795335,1.667876,0.967726,0.861459,0.543210,1.315610
3,1.0,70.04908,Unknown,2.422679,2.258813,0.723078,1.880941,0.761319,2.569422,0.801526,...,0.713647,1.878832,2.196732,1.111966,1.202515,2.758957,1.278797,1.290291,0.863953,1.861841
4,1.0,70.06267,Unknown,2.540917,1.170215,1.034535,1.659528,1.117443,2.654963,1.274813,...,0.537261,1.570875,1.906110,0.870221,0.949140,2.417065,1.161074,1.024230,0.659277,1.554191
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,1.0,732.79951,Unknown,0.506804,0.612721,0.196713,0.892378,0.475693,0.581129,0.890705,...,0.017447,0.489837,0.594974,2.769619,0.122163,0.785278,0.310181,0.159557,2.217542,0.478689
5440,1.0,748.76437,Unknown,0.628861,0.752324,0.284480,0.916754,0.590267,0.713625,1.069844,...,0.189223,0.536927,0.717695,2.766869,0.254884,0.873304,0.465820,0.275240,1.985389,0.530473
5441,1.0,794.79590,Unknown,1.839395,2.536197,2.495466,1.151250,0.635577,0.725910,0.968171,...,2.326771,0.613551,0.719249,3.312826,2.469927,2.195082,2.051681,3.012239,2.383752,0.606691
5442,1.0,800.81295,Unknown,2.041230,3.524770,1.454439,1.478938,1.043713,1.113634,1.284358,...,2.322362,0.940685,0.965685,3.382798,0.657666,1.127591,0.712799,2.344157,2.588098,0.935822


In [6]:
# get metadata
df_join_raw_metadata = df_join_raw.iloc[:, :2]
df_join_raw_metadata

,Average Rt,Average Mz
0,1.0,69.99951
1,1.0,70.04025
2,1.0,70.04151
3,1.0,70.04908
4,1.0,70.06267
...,...,...
5439,1.0,732.79951
5440,1.0,748.76437
5441,1.0,794.79590
5442,1.0,800.81295


In [7]:
# filter by samples
columns_sample = [column for column in df_join_raw.columns if column.split("_")[0] not in groups_id_no]
df_join_raw_intensity = df_join_raw.loc[:, columns_sample]
df_join_raw_intensity = df_join_raw_intensity.iloc[:, 3:]
df_join_raw_intensity

,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,AR_1.8,AR_1.9,AR_1.10,...,SKD_2.43,SKD_2.44,SKD_2.45,SKD_2.46,SKD_2.47,SKD_2.48,SKD_2.49,SKD_2.50,SKD_2.51,SKD_2.52
0,0.786205,0.836262,-0.975441,0.067434,-0.905178,0.458622,-0.756562,-0.691203,-0.867780,0.263093,...,-0.906900,0.153942,0.230629,-0.554414,-0.477171,0.569892,-0.270328,-0.399430,-0.774251,0.137557
1,2.395434,2.572471,2.052794,2.617854,2.349623,2.501793,2.941043,2.046290,2.499543,1.726470,...,4.318123,1.830162,3.076034,2.600924,2.340488,2.455378,2.681824,3.990780,3.177152,2.460345
2,0.564810,1.888370,0.508263,1.088222,0.559000,1.711682,0.639700,0.746858,1.800917,1.991451,...,0.433149,1.329261,1.390541,0.725222,0.795335,1.667876,0.967726,0.861459,0.543210,1.315610
3,2.422679,2.258813,0.723078,1.880941,0.761319,2.569422,0.801526,1.345976,2.523575,2.511583,...,0.713647,1.878832,2.196732,1.111966,1.202515,2.758957,1.278797,1.290291,0.863953,1.861841
4,2.540917,1.170215,1.034535,1.659528,1.117443,2.654963,1.274813,1.073197,1.155376,2.621076,...,0.537261,1.570875,1.906110,0.870221,0.949140,2.417065,1.161074,1.024230,0.659277,1.554191
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,0.506804,0.612721,0.196713,0.892378,0.475693,0.581129,0.890705,3.392578,0.578852,-0.140783,...,0.017447,0.489837,0.594974,2.769619,0.122163,0.785278,0.310181,0.159557,2.217542,0.478689
5440,0.628861,0.752324,0.284480,0.916754,0.590267,0.713625,1.069844,0.272036,0.711182,-0.077285,...,0.189223,0.536927,0.717695,2.766869,0.254884,0.873304,0.465820,0.275240,1.985389,0.530473
5441,1.839395,2.536197,2.495466,1.151250,0.635577,0.725910,0.968171,0.568468,0.723394,0.104266,...,2.326771,0.613551,0.719249,3.312826,2.469927,2.195082,2.051681,3.012239,2.383752,0.606691
5442,2.041230,3.524770,1.454439,1.478938,1.043713,1.113634,1.284358,0.893378,1.111563,0.555390,...,2.322362,0.940685,0.965685,3.382798,0.657666,1.127591,0.712799,2.344157,2.588098,0.935822


In [8]:
df_join_raw_intensity.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5444 entries, 0 to 5443
Columns: 2322 entries, AR_1.1 to SKD_2.52
dtypes: float64(2322)
memory usage: 96.5 MB


In [9]:
check_dataset(df_join_raw_intensity)

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 486125
Count zero:	 85
Count positive:	 12154758
Count greater than 1:	 10317978
Count less than -1:	 60943


### Generate graphs

In [10]:
# Transformation (log10)

if apply_transformation:
	df_join_raw_log = log10_global(df_join_raw_intensity)
else:
	df_join_raw_log = df_join_raw_intensity.copy()
df_join_raw_log.head()

,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,AR_1.8,AR_1.9,AR_1.10,...,SKD_2.43,SKD_2.44,SKD_2.45,SKD_2.46,SKD_2.47,SKD_2.48,SKD_2.49,SKD_2.50,SKD_2.51,SKD_2.52
0,0.786205,0.836262,-0.975441,0.067434,-0.905178,0.458622,-0.756562,-0.691203,-0.867780,0.263093,...,-0.906900,0.153942,0.230629,-0.554414,-0.477171,0.569892,-0.270328,-0.399430,-0.774251,0.137557
1,2.395434,2.572471,2.052794,2.617854,2.349623,2.501793,2.941043,2.046290,2.499543,1.726470,...,4.318123,1.830162,3.076034,2.600924,2.340488,2.455378,2.681824,3.990780,3.177152,2.460345
2,0.564810,1.888370,0.508263,1.088222,0.559000,1.711682,0.639700,0.746858,1.800917,1.991451,...,0.433149,1.329261,1.390541,0.725222,0.795335,1.667876,0.967726,0.861459,0.543210,1.315610
3,2.422679,2.258813,0.723078,1.880941,0.761319,2.569422,0.801526,1.345976,2.523575,2.511583,...,0.713647,1.878832,2.196732,1.111966,1.202515,2.758957,1.278797,1.290291,0.863953,1.861841
4,2.540917,1.170215,1.034535,1.659528,1.117443,2.654963,1.274813,1.073197,1.155376,2.621076,...,0.537261,1.570875,1.906110,0.870221,0.949140,2.417065,1.161074,1.024230,0.659277,1.554191


In [11]:
check_dataset(df_join_raw_log)

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 486125
Count zero:	 85
Count positive:	 12154758
Count greater than 1:	 10317978
Count less than -1:	 60943


In [12]:
# split graph in groups and subgroups

""" def split_groups_subgroups(df_join_raw_log, groups_id, subgroups_id, by_group=False):
	list_df_groups_subgroups = []
	for group in groups_id:
		df_aux = df_join_raw_log.filter(like=group)
		list_aux = []
		
		if by_group:
			list_aux.append(df_aux)
		else:
			for subgroup in subgroups_id[group]:
				list_aux.append(df_aux.filter(like="{}_{}.".format(group, subgroup)))
		list_df_groups_subgroups.append(list_aux)
	return list_df_groups_subgroups """

dict_df_groups_subgroups = split_groups_subgroups(df_join_raw_log, groups_id, subgroups_id)
dict_df_groups_subgroups[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,AR_1.8,AR_1.9,AR_1.10,...,AR_1.221,AR_1.222,AR_1.223,AR_1.224,AR_1.225,AR_1.226,AR_1.227,AR_1.228,AR_1.229,AR_1.230
0,0.786205,0.836262,-0.975441,0.067434,-0.905178,0.458622,-0.756562,-0.691203,-0.867780,0.263093,...,-1.024441,0.182004,-0.806927,0.554472,-0.886178,0.436072,0.503814,-0.893926,0.149128,-0.099162
1,2.395434,2.572471,2.052794,2.617854,2.349623,2.501793,2.941043,2.046290,2.499543,1.726470,...,1.862969,2.722675,2.719321,4.872176,4.804422,2.968291,1.849287,2.376290,2.920405,2.637240
2,0.564810,1.888370,0.508263,1.088222,0.559000,1.711682,0.639700,0.746858,1.800917,1.991451,...,0.480609,1.149052,0.607919,0.657561,1.761644,1.283876,1.952942,1.786393,1.775656,1.226999
3,2.422679,2.258813,0.723078,1.880941,0.761319,2.569422,0.801526,1.345976,2.523575,2.511583,...,2.610493,1.964444,2.347321,2.604101,2.669948,2.149518,2.311077,2.353809,0.798096,1.910588
4,2.540917,1.170215,1.034535,1.659528,1.117443,2.654963,1.274813,1.073197,1.155376,2.621076,...,2.695032,1.742477,2.554994,1.294899,2.772503,1.920325,0.971670,1.124530,1.267771,1.577614


In [13]:
check_dataset(dict_df_groups_subgroups[groups_id[0]][subgroups_id[groups_id[0]][0]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 60081
Count zero:	 6
Count positive:	 1192033
Count greater than 1:	 1030593
Count less than -1:	 8283


In [14]:
# No apply transpose for kneighbors_graph
# dict_groups_subgroups_t = dict_df_groups_subgroups.copy()

# Aplly Transpose
dict_groups_subgroups_t = transpose_global(dict_df_groups_subgroups)

dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,0,1,2,3,4,5,6,7,8,9,...,5434,5435,5436,5437,5438,5439,5440,5441,5442,5443
0,0.786205,2.395434,0.564810,2.422679,2.540917,2.551968,2.672494,2.686214,2.670733,2.725781,...,2.630707,1.375523,-0.374743,1.174494,0.839537,0.506804,0.628861,1.839395,2.041230,0.058853
1,0.836262,2.572471,1.888370,2.258813,1.170215,1.283065,2.159618,1.491081,1.443890,1.510544,...,3.781503,3.727650,2.120324,3.272872,2.994812,0.612721,0.752324,2.536197,3.524770,0.167473
2,-0.975441,2.052794,0.508263,0.723078,1.034535,1.155696,1.377911,2.538766,2.559140,2.669594,...,2.648348,1.134116,2.200028,0.988740,0.544839,0.196713,0.284480,2.495466,1.454439,-0.256118
3,0.067434,2.617854,1.088222,1.880941,1.659528,1.767491,2.008215,2.092772,2.072683,2.185935,...,2.207700,1.753761,0.217247,1.533782,1.191297,0.892378,0.916754,1.151250,1.478938,0.344954
4,-0.905178,2.349623,0.559000,0.761319,1.117443,1.233731,1.445150,1.451650,1.403755,1.476032,...,1.600030,1.351677,-0.412769,1.145622,0.814454,0.475693,0.590267,0.635577,1.043713,0.021154


In [15]:
check_dataset(dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 60081
Count zero:	 6
Count positive:	 1192033
Count greater than 1:	 1030593
Count less than -1:	 8283


In [16]:
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf
from sklearn.neighbors import kneighbors_graph

def correlation_ledoitwolf_global(exp, dict_groups_subgroups_t):
	dict_groups_subgroups_t_corr = {}
	for group_id, dict_groups in dict_groups_subgroups_t.items():
		dict_aux = {}
		for subgroup_id, df_subgroup in dict_groups.items():
			print(group_id, subgroup_id, df_subgroup.shape)
			
			""" import numpy as np
			cov = np.cov(df_subgroup.values, rowvar=False)
			cond = np.linalg.cond(cov)
			print("Condition number:", cond, cond > 1e8) # ill-conditioned if > 1e8 (True, instable) """

			scaler = StandardScaler()
			df_subgroup_scaled = scaler.fit_transform(df_subgroup)
			lw = LedoitWolf()
			lw.fit(df_subgroup_scaled)

			# Matriz de covarianza regularizada
			cov = lw.covariance_
			std = np.sqrt(np.diag(cov))
			corr = cov / np.outer(std, std)
			matrix = pd.DataFrame(corr)
		
			dict_aux[subgroup_id] = matrix
			
			matrix.to_csv("experiments/output/{}/correlations/{}_{}.csv".format(exp, group_id, subgroup_id), index=True)
		dict_groups_subgroups_t_corr[group_id] = dict_aux
	return dict_groups_subgroups_t_corr

def correlation_kneighbors_graph_global(exp, dict_groups_subgroups_t):
	dict_groups_subgroups_t_corr = {}
	for group_id, dict_groups in dict_groups_subgroups_t.items():
		dict_aux = {}
		for subgroup_id, df_subgroup in dict_groups.items():
			print(group_id, subgroup_id, df_subgroup.shape)
			
			""" import numpy as np
			cov = np.cov(df_subgroup.values, rowvar=False)
			cond = np.linalg.cond(cov)
			print("Condition number:", cond, cond > 1e8) # ill-conditioned if > 1e8 (True, instable) """
			
			k = 10
			scaler = StandardScaler()
			df_subgroup_scaled = scaler.fit_transform(df_subgroup)
			A = kneighbors_graph(
				df_subgroup_scaled, # X, X_scaled
				n_neighbors=k,
				metric="euclidean", # "cosine",
				mode="distance",
				include_self=True
			)
			matrix = pd.DataFrame(A.toarray())
		
			dict_aux[subgroup_id] = matrix
			
			matrix.to_csv("experiments/output/{}/correlations/{}_{}.csv".format(exp, group_id, subgroup_id), index=True)
		dict_groups_subgroups_t_corr[group_id] = dict_aux
	return dict_groups_subgroups_t_corr

In [17]:
# Correlation matrix (partial correlation)

# Option 1
# dict_groups_subgroups_t_corr = correlation_global(exp, dict_groups_subgroups_t)

# Option 2
dict_groups_subgroups_t_corr = correlation_ledoitwolf_global(exp, dict_groups_subgroups_t)

# Option 3
# dict_groups_subgroups_t_corr = correlation_kneighbors_graph_global(exp, dict_groups_subgroups_t)

dict_groups_subgroups_t_corr[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

AR 1 (230, 5444)
AR 2 (230, 5444)
BC 1 (41, 5444)
BC 2 (41, 5444)
BPH 1 (39, 5444)
BPH 2 (39, 5444)
CKD-Mild 1 (116, 5444)
CKD-Mild 2 (116, 5444)
CKD-Moderate 1 (60, 5444)
CKD-Moderate 2 (60, 5444)
CKD-Severe 1 (89, 5444)
CKD-Severe 2 (89, 5444)
CKD5-HD 1 (42, 5444)
CKD5-HD 2 (42, 5444)
CKD5-PD 1 (37, 5444)
CKD5-PD 2 (37, 5444)
CRS 1 (14, 5444)
CRS 2 (14, 5444)
Health 1 (120, 5444)
Health 2 (120, 5444)
LPRD 1 (11, 5444)
LPRD 2 (11, 5444)
LSNB 1 (34, 5444)
LSNB 2 (34, 5444)
OSA 1 (58, 5444)
OSA 2 (58, 5444)
PCa 1 (53, 5444)
PCa 2 (53, 5444)
PD 1 (88, 5444)
PD 2 (88, 5444)
RCC 1 (71, 5444)
RCC 2 (71, 5444)
SGB 1 (43, 5444)
SGB 2 (43, 5444)
SKD 1 (52, 5444)
SKD 2 (52, 5444)


,0,1,2,3,4,5,6,7,8,9,...,5434,5435,5436,5437,5438,5439,5440,5441,5442,5443
0,1.000000,0.066367,0.084551,0.063088,0.026547,0.107003,0.114833,0.081186,0.028873,0.097727,...,0.028606,0.013613,0.042491,-0.048360,0.063541,-0.003766,0.116792,0.024187,0.024305,0.103877
1,0.066367,1.000000,0.009390,0.177361,0.312923,0.339069,0.351477,0.263944,0.271674,0.239090,...,0.211515,0.201298,0.059842,0.160483,0.181520,0.194460,0.277337,0.187480,0.231946,0.237027
2,0.084551,0.009390,1.000000,-0.007082,-0.030890,-0.012020,-0.030460,-0.040366,-0.035757,-0.006362,...,0.109535,0.085926,0.066003,0.053312,0.095418,-0.004039,0.050499,0.061673,0.087238,0.070281
3,0.063088,0.177361,-0.007082,1.000000,0.424128,0.394625,0.364259,0.395826,0.369737,0.398126,...,0.224266,0.253471,0.057056,0.120968,0.174269,0.137434,0.178985,0.253257,0.254985,0.218372
4,0.026547,0.312923,-0.030890,0.424128,1.000000,0.796617,0.707197,0.665279,0.638566,0.577991,...,0.186247,0.154169,-0.102948,0.039029,0.163326,0.105520,0.175044,0.155146,0.163970,0.171869


In [18]:
# Check correlation matrices

dict_groups_subgroups_t_corr

{'AR': {'1':           0         1         2         3         4         5         6     \
  0     1.000000  0.066367  0.084551  0.063088  0.026547  0.107003  0.114833   
  1     0.066367  1.000000  0.009390  0.177361  0.312923  0.339069  0.351477   
  2     0.084551  0.009390  1.000000 -0.007082 -0.030890 -0.012020 -0.030460   
  3     0.063088  0.177361 -0.007082  1.000000  0.424128  0.394625  0.364259   
  4     0.026547  0.312923 -0.030890  0.424128  1.000000  0.796617  0.707197   
  ...        ...       ...       ...       ...       ...       ...       ...   
  5439 -0.003766  0.194460 -0.004039  0.137434  0.105520  0.103135  0.085186   
  5440  0.116792  0.277337  0.050499  0.178985  0.175044  0.184306  0.126516   
  5441  0.024187  0.187480  0.061673  0.253257  0.155146  0.128687  0.163209   
  5442  0.024305  0.231946  0.087238  0.254985  0.163970  0.126780  0.138110   
  5443  0.103877  0.237027  0.070281  0.218372  0.171869  0.163607  0.135128   
  
            7         8   

In [19]:
check_dataset(dict_groups_subgroups_t_corr[groups_id[0]][subgroups_id[groups_id[0]][1]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 5625942
Count zero:	 4878
Count positive:	 24006316
Count greater than 1:	 0
Count less than -1:	 0


In [20]:
def build_graph_weight_global_directed_new(exp, dict_groups_subgroups_t_corr, threshold=0.3):
	dict_groups_subgroups_t_corr_g = {}
	for group_id, dict_groups in dict_groups_subgroups_t_corr.items():
		dict_aux = {}
		for subgroup_id, df_subgroup in dict_groups.items():
			# Percentil Correlaciones conservadas 	Densidad
			# 50	    50 %	                    Muy densa
            # 75	    25 %	                    Densa
            # 90	    10 %	                    Moderadamente dispersa
            # 95	    5 %	                        Dispersa
            # 99	    1 %	                        Muy dispersa
			threshold = np.percentile(np.abs(df_subgroup), 90)
			
			df_weighted_edges = (df_subgroup.where(np.triu(np.ones(df_subgroup.shape), k=1).astype(bool)).stack())
			df_weighted_edges = df_weighted_edges.dropna().to_frame()
			df_weighted_edges.reset_index(inplace=True)
			df_weighted_edges.columns = ["source", "target", "weight"]
			df_weighted_edges = df_weighted_edges[df_weighted_edges["weight"].abs() >= threshold]
			df_weighted_edges["subgroup"] = [subgroup_id] * len(df_weighted_edges)
			dict_aux[subgroup_id] = df_weighted_edges
			
			df_weighted_edges.to_csv("experiments/output/{}/preprocessing/edges/{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# G = nx.from_pandas_edgelist(df_weighted_edges, "source", "target", edge_attr=["weight"])
			# print(groups_id[i], subgroups_id[groups_id[i]][j], G.number_of_nodes(), G.number_of_edges())
			# nx.write_gexf(G, "experiments/output/{}/preprocessing/graphs/graphs_{}_{}.gexf".format(exp, groups_id[i], subgroups_id[groups_id[i]][j]))
		dict_groups_subgroups_t_corr_g[group_id] = dict_aux
	return dict_groups_subgroups_t_corr_g

In [21]:
# Build graph (corpus graphs)

# dict_groups_subgroups_t_corr_g = build_graph_weight_global_directed(exp, dict_groups_subgroups_t_corr, threshold=threshold_corr)
dict_groups_subgroups_t_corr_g = build_graph_weight_global_directed_new(exp, dict_groups_subgroups_t_corr, threshold=threshold_corr)
dict_groups_subgroups_t_corr_g[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,source,target,weight,subgroup
49,0,50,0.499294,1
50,0,51,0.471649,1
58,0,59,0.430840,1
63,0,64,0.582978,1
67,0,68,0.501214,1


In [22]:
def create_graph_data_directed_features(exp, groups_id, subgroups_id, dict_df_groups_subgroups, df_join_raw_metadata):	
	for group_id in tqdm(groups_id):
		for subgroup_id in tqdm(subgroups_id[group_id]):
			df_weighted_edges = pd.read_csv("experiments/output/{}/preprocessing/edges/{}_{}.csv".format(exp, group_id, subgroup_id))
			# print(df_weighted_edges)
			G = nx.from_pandas_edgelist(df_weighted_edges, "source", "target", edge_attr=["weight", "subgroup"])
			dict_id_idx = dict(zip(list(G.nodes()), range(G.number_of_nodes())))
			G = nx.relabel_nodes(G, dict_id_idx)

			df_nodes = dict_df_groups_subgroups[group_id][subgroup_id].loc[list(dict_id_idx.keys())] # A_1.1, A_1.2, A_1.3
			# from IPython.display import display
			# display(df_nodes)

			# nodes, with node features
			metadata = df_join_raw_metadata.loc[df_nodes.index] # Average Rt, Average Mz
			# intensity = df_join_raw_log.loc[df_nodes.index] # A_1.1, A_1.2, A_1.3, A_2.1, ...

			""" e = 1e-8
			mz = metadata.iloc[:, 1].values
			rt = metadata.iloc[:, 0].values
			intensity_mean = df_nodes.mean(axis=1).values
			intensity_std = df_nodes.std(axis=1).values
			intensity_cv = intensity_std / intensity_mean
			presence_ratio = (df_nodes > 0).mean(axis=1)

			mz_log = np.log10(mz + e)
			# intensity_mean_log = np.log10(intensity_mean + e)
			
			# mz_z = (mz_log - mz_log.mean()) / mz_log.std() # z-score
			rt_z = (rt - rt.mean()) / rt.std() # z-score
			# intensity_mean_z = (intensity_mean_log - intensity_mean_log.mean()) / intensity_mean_log.std() # z-score

			data_node = {
				"idx": list(dict_id_idx.values()),
				"id": list(dict_id_idx.keys()),
				"mz": mz_log,
				"rt": rt_z,
				"intensity_mean": intensity_mean,
				"intensity_std": intensity_std,
				"intensity_cv": intensity_cv,
				"presence_ratio": presence_ratio
			}
			for i in range(len(df_nodes.columns)):
				data_node[i] = df_nodes.iloc[:, i]

			df_node_features = pd.DataFrame(data_node)
			# df_node_features.insert(0, "idx", list(dict_id_idx.values()))
			# df_node_features.insert(1, "id", list(dict_id_idx.keys()))
			df_node_features.to_csv("experiments/output/{}/preprocessing/graphs_data/nodes_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# print(df_node_features) """

			data_node = {
				"idx": list(dict_id_idx.values()),
				"id": list(dict_id_idx.keys()),
				"mz": metadata.iloc[:, 1].values,
				"rt": metadata.iloc[:, 0].values,
			}
			for i in range(len(df_nodes.columns)):
				data_node[i] = df_nodes.iloc[:, i]

			df_node_features = pd.DataFrame(data_node)
			df_node_features.to_csv("experiments/output/{}/preprocessing/graphs_data/nodes_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# print(df_node_features)

			# edges
			edges = list(G.edges())
			df_edges = pd.DataFrame(edges, columns=["source", "target"])
			df_edges["weight"] = [G.get_edge_data(*edge)["weight"] for edge in edges]
			df_edges["subgroup"] = [G.get_edge_data(*edge)["subgroup"] for edge in edges]
			df_edges.to_csv("experiments/output/{}/preprocessing/graphs_data/edges_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)


In [23]:
# create dataset - nodes/edge data for PyTorch Geometric/DGL framework

# IMPORTANT
dict_df_groups_subgroups_ = split_groups_subgroups(df_join_raw_intensity, groups_id, subgroups_id) # Important (intesities without Log)

for data_variation in data_variations:
	if data_variation == "none":
		create_graph_data_directed_features(exp, groups_id, subgroups_id, dict_df_groups_subgroups_, df_join_raw_metadata)
	else:
		# dynamic graph to static graph
		create_graph_data_directed_variation(exp, groups_id, subgroups_id, dict_df_groups_subgroups_, data_variation)

100%|██████████| 18/18 [06:01<00:00, 20.06s/it]


In [24]:
# details
list_details = []
	
for group_id in groups_id:
	subgroups_id_ = []
	for data_variation in data_variations:
		if data_variation == "none":
			subgroups_id_ += subgroups_id[group_id]
		else:
			subgroups_id_ += [data_variation]
	# print(subgroups)
	
	for subgroup_id_ in subgroups_id_:
		try:
			df_edges = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/edges_{}_{}.csv".format(exp, group_id, subgroup_id_))

			G = nx.from_pandas_edgelist(df_edges.iloc[:, [0, 1]])
			list_details.append([group_id, subgroup_id_, G.number_of_nodes(), G.number_of_edges(), nx.density(G), np.nan, nx.is_connected(G)])
		except:
			list_details.append([group_id, subgroup_id_, G.number_of_nodes(), G.number_of_edges(), np.nan, np.nan, np.nan])

df_details = pd.DataFrame(list_details, columns=["Group", "Subgroup", "Num. nodes", "Num. edges", "Density", "Diameter", "Is connected"])
df_details.to_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp), index=False)

df_details = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp))
df_details

,Group,Subgroup,Num. nodes,Num. edges,Density,Diameter,Is connected
0,AR,1,5339,1479135,0.103800,NaN,False
1,AR,2,5381,1479135,0.102186,NaN,False
2,BC,1,5444,1479135,0.099835,NaN,True
3,BC,2,5444,1479135,0.099835,NaN,True
4,BPH,1,5444,1479135,0.099835,NaN,True
5,BPH,2,5444,1479135,0.099835,NaN,True
6,CKD-Mild,1,5443,1479135,0.099871,NaN,True
7,CKD-Mild,2,5441,1479135,0.099945,NaN,True
8,CKD-Moderate,1,5444,1479135,0.099835,NaN,True
9,CKD-Moderate,2,5444,1479135,0.099835,NaN,True
